# Confidence calibration

How well the model's stated `confidence` matches how often it is right, for one `evaluate` sweep.

After https://arxiv.org/pdf/2512.16030

In [2]:
%matplotlib ipympl

In [5]:
from analysis.build_data import load_trace_df
from analysis.calibration import (
    category_accuracy_table,
    category_table,
    confidence_metrics,
    expected_calibration_error,
    horizon_table,
    kalshibench_metrics,
)


In [ ]:
# one scoreable forecast per row, from every child run of the sweep (see analysis/build_data.py)
forecasts = load_trace_df("outcome-confidence", run_name="2k-balanced")

## Dataset statistics

Table 2 of the paper: the markets in the sample by category, with the share resolving to the priced outcome (see `category_table` in `analysis/calibration.py`).

In [30]:
category_table(forecasts).round(1)

,N,%,Yes%
category,,,
economics,753,29.3,42.0
elections,582,22.7,27.1
finance,433,16.9,43.4
companies,400,15.6,51.2
politics,400,15.6,25.0


## KalshiBench table

Table 4 of the paper, per model (see `kalshibench_metrics` in `analysis/calibration.py` for the columns).

In [33]:
model_table = forecasts.groupby("model").apply(
    kalshibench_metrics, include_groups=False
)

In [34]:
model_table.round(3)

,N,Acc,F1,F1yes,Brier,BSS,ECE,MCE
model,,,,,,,,
claude-fable-5-1,543.0,65.930,0.531,0.286,0.211,0.106,0.031,0.062
claude-opus-4-5,1208.0,65.315,0.633,0.547,0.221,0.086,0.059,0.400
claude-opus-5-5,1619.0,70.290,0.626,0.456,0.182,0.194,0.021,0.064


## Confidence analysis

Table 5 of the paper, per model: average confidence, confidence when wrong, and the overconfidence rate above 70/80/90% (see `confidence_metrics` in `analysis/calibration.py`).

In [35]:
confidence_table = forecasts.groupby("model").apply(
    confidence_metrics, include_groups=False
)

In [36]:
confidence_table.round(1)

,Avg Conf,Conf_wrong,OCR@70,OCR@80,OCR@90
model,,,,,
claude-fable-5-1,64.1,59.1,12.7,11.5,8.8
claude-opus-4-5,69.0,65.3,26.3,17.2,15.3
claude-opus-5-5,70.6,62.1,13.1,8.4,5.9


In [37]:
# the reliability table behind the number: per bin, its size, conf(B_b) and acc(B_b)
reliability = forecasts.groupby("confidence_bin", observed=True).agg(
    n=("correct", "size"), conf=("confidence", "mean"), acc=("correct", "mean")
)

## Forecast horizon

Accuracy and confidence by days from the model's training cutoff to the market's close (see `horizon_table` in `analysis/calibration.py`).

In [38]:
horizon_table(forecasts).round(2)

N    Acc  Avg Conf   ECE
model                                                      
claude-fable-5-1 (30.0, 60.0]     18  88.89     79.33  0.16
                 (60.0, 90.0]     41  82.93     80.30  0.15
                 (90.0, 180.0]   482   63.9     62.13  0.04
                 (180.0, 365.0]    1    0.0     93.00  0.93
                 (365.0, inf]      1    0.0     70.00  0.70
claude-opus-4-5  (-0.001, 30.0]    8   62.5     74.62  0.26
                 (30.0, 60.0]     23  86.96     92.57  0.08
                 (60.0, 90.0]     42  59.52     81.76  0.22
                 (90.0, 180.0]    83   74.7     79.86  0.07
                 (180.0, 365.0]  461  59.87     67.14  0.11
                 (365.0, inf]    591  67.85     66.96  0.03
claude-opus-5-5  (-0.001, 30.0]  115  85.22     77.19  0.09
                 (30.0, 60.0]    215  77.21     75.47  0.05
                 (60.0, 90.0]    494   66.8     71.64  0.06
                 (90.0, 180.0]   784  68.62     67.50  0.02
                 (180.0, 365.0]    6   50.0     86.00  0.48
                 (365.0, inf]      5   60.0     70.20  0.41

## Market category

Accuracy and confidence by model and market category (see `category_accuracy_table` in `analysis/calibration.py`).

In [39]:
category_accuracy_table(forecasts).round(2)

N    Acc  Avg Conf   ECE
model            category                             
claude-fable-5-1 companies   19  68.42     73.32  0.18
                 economics  397  61.46     58.98  0.04
                 elections  127  79.53     78.91  0.06
claude-opus-4-5  companies  400  58.75     73.34  0.15
                 economics  373  60.32     64.69  0.08
                 elections  400   77.0     68.98  0.09
                 finance     35   60.0     64.37  0.13
claude-opus-5-5  companies   19  68.42     70.95  0.25
                 economics  400  61.25     57.20  0.05
                 elections  400   77.5     78.02  0.04
                 finance    400   63.5     67.89  0.05
                 politics   400   79.0     79.22  0.06